# Startup Funding Prediction

Train and evaluate the funding success classifier and funding amount regressor using the bundled dataset or a custom CSV.

In [ ]:
from pathlib import Path

import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, mean_absolute_error, r2_score

from model import (
    load_dataset,
    train_models,
    display_top_drivers,
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    SAMPLE_PREVIEW,
    DATA_PATH,
)

# Set this to override the dataset location if needed
DATA_FILE = DATA_PATH  # Path("/path/to/startup_funding.csv")

In [ ]:
# Load data and prepare features/targets
df = load_dataset(DATA_FILE)
feature_frame = df[CATEGORICAL_FEATURES + NUMERIC_FEATURES]

artifacts = train_models(feature_frame, df["amount_usd"])
classifier = artifacts.classifier
regressor = artifacts.regressor
split = artifacts.split

In [ ]:
# Evaluation metrics
success_pred = classifier.predict(split.X_test)
success_proba = classifier.predict_proba(split.X_test)[:, 1]
print("Classification metrics (above-median flag):")
print(classification_report(split.success_test, success_pred, digits=3))
print(f"ROC AUC: {roc_auc_score(split.success_test, success_proba):.3f}")

log_amount_pred = regressor.predict(split.X_test)
amount_pred = np.expm1(log_amount_pred)
true_amount = np.expm1(split.log_amount_test)
mae = mean_absolute_error(true_amount, amount_pred)
r2 = r2_score(true_amount, amount_pred)
print("\nRegression metrics (expected funding amount):")
print(f"MAE (USD): {mae:,.0f}")
print(f"R^2: {r2:.3f}")

display_top_drivers(classifier)

In [ ]:
# Sample predictions
sample_size = min(SAMPLE_PREVIEW, len(split.X_test))
sample_output = split.X_test.iloc[:sample_size].copy()
sample_output["success_probability"] = success_proba[:sample_size]
sample_output["predicted_amount_usd"] = amount_pred[:sample_size]

sample_output[
    [
        "startup_name",
        "industry_vertical",
        "city_location",
        "investment_type",
        "success_probability",
        "predicted_amount_usd",
    ]
].sort_values(by="success_probability", ascending=False).reset_index(drop=True)